In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("src")

import torch
import gc
import random
import pandas as pd
import re
from tqdm import tqdm

import _dataset
import _prompt
import _mapping
import _util
from _intervention import get_label_probability

In [3]:
model, tokenizer = _util.load_OSS()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [9]:
prompts = pd.read_csv("data/pre_result_prompts.csv")
prompts["base_sum"] = prompts["base_sum"].astype('Int64')
prompts["source_sum"] = prompts["source_sum"].astype('Int64')
print(f"loaded {len(prompts)} divided prompts")

loaded 128 divided prompts


In [16]:
intervention_ids = _mapping.full_stepwise_intervene_loc_3_digit["restatement"]

In [17]:
# Get header of divided prompts dataset
header = list(prompts.columns) + ['factual_label_probability', 'counterfactual_label_probability']

filepath = _util.create_csv_file("experiments/token_intervention/output/GPT_OSS_stepwise/probability", "probability_pre_result_restatement.csv", header, overwrite=True)

batch_size = 16

for i in tqdm(range(0, len(prompts), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = prompts.iloc[i:i+batch_size]

    factual_labels = batch_rows['base_sum']
    
    
    # Prepare batch of intervention prompts
    intervention_prompts = [_prompt.get_intervened_prompt(intervention_ids, row['base_prompt'], row['source_prompt']) for _, row in batch_rows.iterrows()]
    
    # Tokenize all prompts in the batch
    tokens = tokenizer(intervention_prompts, return_tensors="pt", padding=True, padding_side="left").to(model.device)
    
    base_labels = tokenizer([str(base_sum) for base_sum in batch_rows['base_sum']], return_tensors="pt", padding=True, padding_side="right")["input_ids"].to(model.device)
    base_labels_mask = base_labels != tokenizer.pad_token_id
    base_labels_probs = get_label_probability(model, tokens, base_labels, base_labels_mask)

    source_labels = tokenizer([str(source_sum) for source_sum in batch_rows['source_sum']], return_tensors="pt", padding=True, padding_side="right")["input_ids"].to(model.device)
    source_labels_mask = source_labels != tokenizer.pad_token_id
    source_labels_probs = get_label_probability(model, tokens, source_labels, source_labels_mask)
    
    # Process each generated text in the batch
    for j, (_, row) in enumerate(batch_rows.iterrows()):
        _util.write_to_csv(filepath, row.to_list() + [base_labels_probs[j].item(), source_labels_probs[j].item()])


  0%|                                                                                                     | 0/8 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████████████████████████████████████████████████████| 8/8 [02:06<00:00, 15.85s/it]


In [ ]:
# Load the divided prompts dataset
divided_prompts = pd.read_csv("data/hundreds_pre_result_divided_prompts.csv")
divided_prompts["base_number"] = divided_prompts["base_number"].astype('Int64')
divided_prompts["source_number"] = divided_prompts["source_number"].astype('Int64')
print(f"loaded {len(divided_prompts)} divided prompts")

loaded 128 prompts


In [ ]:
# Get header of divided prompts dataset
header = list(divided_prompts.columns) + ['factual_label_probability', 'source_label_probability']

filepath = _util.create_csv_file("experiments/behavioral_accuracy/output/GPT_OSS", "restatement_source_intervention.csv", header)

batch_size = 16

for i in tqdm(range(0, len(divided_prompts), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = divided_prompts.iloc[i:i+batch_size]

    factual_labels = batch_rows['base_sum']
    
    
    # Prepare batch of intervention prompts
    intervention_prompts = [_prompt.get_intervened_prompt(intervention_ids, row['base_prompt'], row['source_prompt']) for _, row in batch_rows.iterrows()]
    
    # Tokenize all prompts in the batch
    tokens = tokenizer(intervention_prompts, return_tensors="pt", padding=True, padding_side="left").to(model.device)
    
    base_labels = tokenizer([str(base_sum) for base_sum in batch_rows['base_sum']], return_tensors="pt", padding=True, padding_side="right").to(model.device)
    base_labels_mask = base_labels != tokenizer.pad_token_id
    base_labels_probs = get_label_probability(model, tokens, base_labels, base_labels_mask)

    source_labels = tokenizer([str(source_sum) for source_sum in batch_rows['counterfactual_sum']], return_tensors="pt", padding=True, padding_side="right").to(model.device)
    source_labels_mask = source_labels != tokenizer.pad_token_id
    source_labels_probs = get_label_probability(model, tokens, source_labels, source_labels_mask)
    
    # Process each generated text in the batch
    for j, (_, row) in enumerate(batch_rows.iterrows()):
        _util.write_to_csv(filepath, row.to_list() + [base_labels_probs[j], source_labels_probs[j]])


 ... (more hidden) ...

 ... (more hidden) ...
